# 9.3절 경험 재현과 CartPole 학습곡선 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter09_3_experience_replay_cartpole.ipynb)

책 본문: [Chapter 9](https://smhanlab.com/book-ml/kor/ml2/chapter09.html)

이 노트북은 본문의 DQN을 **CartPole**에서 그대로 실행합니다. 9.2절의 **타겟
네트워크**와 이 절의 **경험 재현 버퍼**가 한 학습 루프에서 함께 일하면서
"이론적 장치"가 **실제 학습곡선**으로 이어지는 것을 확인합니다 — 먼저 버퍼의
FIFO·무작위 샘플링 동작을 손으로 추적하고, 300에피소드 × 3시드로 학습해
본문의 숫자·그림을 재현합니다.

## 0. 세팅 (폰트, 랜덤시드, 환경)

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import random
from collections import deque
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("torch:", torch.__version__, " gymnasium:", gym.__version__)
print("한글 폰트:", kr[0] if kr else "NOT FOUND — 한글 표시가 깨질 수 있음")


torch: 2.13.0+cpu  gymnasium: 1.3.0
한글 폰트: Noto Sans CJK KR


## 1. ReplayBuffer: FIFO와 무작위 샘플링, 두 동작

본문 "손으로 한 번"에서 추적한 것 그대로: 버퍼는 **FIFO 큐**이고,
`push`는 가득 차면 가장 오래된 항목을 밀어내며, `sample`은 **무작위 비복원
추출**로 `batch_size`개를 뽑는다. `capacity=4`로 동작을 확인하고, 복원추출
12번의 빈도 분포를 본다.

In [2]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)   # maxlen 초과 시 가장 오래된 항목 자동 밀려남(FIFO)

    def push(self, transition):
        self.buffer.append(transition)

    def __len__(self):
        return len(self.buffer)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)   # 무작위 비복원 추출

# --- FIFO: capacity=4 버퍼에 5개를 순서대로 push하면 e1이 밀려난다 ---
buf = ReplayBuffer(4)
for e in ["e1", "e2", "e3", "e4", "e5"]:
    buf.push(e)
    print(f"push {e} -> {list(buf.buffer)}")
assert list(buf.buffer) == ["e2", "e3", "e4", "e5"], "FIFO 밀려남 실패"
print("버퍼가 가득 차면 가장 오래된 항목이 밀려남: OK\n")

# --- 무작위 샘플링: 복원추출 12번(시드 42)의 빈도 분포 ---
random.seed(42)
draws = [random.choice(list(buf.buffer)) for _ in range(12)]
print("복원추출 12번 (시드 42):", draws)
from collections import Counter
print("빈도:", dict(sorted(Counter(draws).items())))

# --- 실제 DQN은 random.sample(비복원): 한 배치 안에 중복 없음 ---
random.seed(0)
batch = buf.sample(3)
print("비복원 3개 배치:", batch)
assert len(set(batch)) == 3, "배치 안에 중복이 있음(비복원이어야 함)"


push e1 -> ['e1']
push e2 -> ['e1', 'e2']
push e3 -> ['e1', 'e2', 'e3']
push e4 -> ['e1', 'e2', 'e3', 'e4']
push e5 -> ['e2', 'e3', 'e4', 'e5']
버퍼가 가득 차면 가장 오래된 항목이 밀려남: OK

복원추출 12번 (시드 42): ['e2', 'e2', 'e4', 'e3', 'e3', 'e3', 'e2', 'e2', 'e5', 'e2', 'e2', 'e2']
빈도: {'e2': 7, 'e3': 3, 'e4': 1, 'e5': 1}
비복원 3개 배치: ['e5', 'e3', 'e2']


## 2. DQN 전체 조립: Q-네트워크 + 손실(타겟 네트워크) + 학습 루프

본문의 세 조각 — 9.1절 `QNetwork`, 9.2절 `dqn_loss`(타겟 네트워크
`θ⁻`로 목표값 계산, Huber 손실), 9.3절 `ReplayBuffer` — 를 하나로 조립한다.
설정: Adam `lr=1e-3`, `batch=32`, `γ=0.99`, 타겟 하드 동기화
`update_freq=100`, `capacity=10000`, **워밍업 1,000스텝**(버퍼가 1,000개
차기 전까지는 학습 없이 경험만 수집), `ε`은 스텝당 0.9997배로 1.0→0.05
감쇠. 학습 루프의 ①push(매 스텝) ②샘플링(워밍업 후 매 스텝)
③손실·역전파 ④100스텝마다 타겟 동기화 순서다.

In [3]:
GAMMA, BATCH, WARMUP, CAPACITY = 0.99, 32, 1000, 10000
EPS_START, EPS_END, EPS_STEP_DECAY = 1.0, 0.05, 0.9997
LR, EPISODES, UPDATE_FREQ = 1e-3, 300, 100

class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )
    def forward(self, x):
        return self.net(x)

def dqn_loss(Q_net, target_net, states, actions, rewards, next_states, dones, gamma):
    q_pred = Q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
    with torch.no_grad():                        # 목표는 이 스텝에서 상수
        q_next = target_net(next_states).max(dim=1).values
        target = rewards + gamma * q_next * (1 - dones)
    return F.smooth_l1_loss(q_pred, target)      # Huber loss

def train_dqn(seed, episodes=EPISODES):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env = gym.make("CartPole-v1")
    Q_net = QNetwork(4, 2)
    target_net = QNetwork(4, 2)
    target_net.load_state_dict(Q_net.state_dict())   # 처음엔 둘이 동일
    opt = torch.optim.Adam(Q_net.parameters(), lr=LR)   # Q_net만 학습!
    buffer = ReplayBuffer(CAPACITY)
    rets, losses, total_step, best = [], [], 0, 0.0
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + 1000 * ep)
        done, ret = False, 0.0
        while not done:
            eps = max(EPS_END, EPS_START * EPS_STEP_DECAY ** total_step)
            a = (random.randrange(env.action_space.n) if random.random() < eps   # 탐색(시드 고정 → 재현 가능)
                 else int(Q_net(torch.tensor(obs, dtype=torch.float32)).argmax()))
            ns, r, term, trunc, _ = env.step(a)
            buffer.push((obs, a, r, ns, float(term)))          # ① push: 매 스텝 1개
            done = term or trunc
            obs, ret, total_step = ns, ret + r, total_step + 1
            if len(buffer) >= WARMUP:                          # ②③ 워밍업 이후 매 스텝 학습
                batch = buffer.sample(BATCH)
                s = torch.tensor(np.array([b[0] for b in batch], dtype=np.float32))
                at = torch.tensor([b[1] for b in batch])
                rr = torch.tensor([b[2] for b in batch], dtype=torch.float32)
                n_ = torch.tensor(np.array([b[3] for b in batch], dtype=np.float32))
                dn = torch.tensor([b[4] for b in batch], dtype=torch.float32)
                loss = dqn_loss(Q_net, target_net, s, at, rr, n_, dn, GAMMA)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(Q_net.parameters(), 10.0)
                opt.step()
                losses.append(loss.item())
            if total_step % UPDATE_FREQ == 0:                  # ④ 100스텝마다 타겟 하드 동기화
                target_net.load_state_dict(Q_net.state_dict())
        rets.append(ret)
        best = max(best, ret)
    env.close()
    return dict(rets=np.array(rets), losses=np.array(losses), best=best, Q_net=Q_net)


## 3. 300에피소드 × 3시드로 학습

초기화, `ε`-탐색, 버퍼 샘플링 — 세 곳의 무작위성 모두 같은 시드로 고정했으므로
**실행이 정확히 재현**된다. 시드당 약 1~2분. 학습 도중 **최고 리턴 500**
(= 500스텝 버티기, 타임아웃 만점)에 도달하는지 확인한다.

In [4]:
results = {seed: train_dqn(seed) for seed in (0, 1, 2)}
for seed, res in results.items():
    print(f"시드 {seed}: 처음 20평균 {np.mean(res['rets'][:20]):.2f} -> 마지막 20평균 {np.mean(res['rets'][-20:]):.2f}, "
          f"최고 {res['best']:.0f}, 손실(마지막 50평균) {np.mean(res['losses'][-50:]):.4f}")


시드 0: 처음 20평균 21.15 -> 마지막 20평균 103.85, 최고 500, 손실(마지막 50평균) 0.6831
시드 1: 처음 20평균 22.10 -> 마지막 20평균 18.70, 최고 500, 손실(마지막 50평균) 1.4333
시드 2: 처음 20평균 21.40 -> 마지막 20평균 140.85, 최고 500, 손실(마지막 50평균) 0.5820


**본문 숫자와 대조** (시드 0):
처음 20에피소드 평균 리턴 ≈ **21.15**, 마지막 20에피소드 ≈ **103.85**,
최고 **500**. 세 시드 모두 학습 도중 500에 도달하지만 마지막 20평균이
19~141로 크게 다름 — 본문 "여러 시드" 절의 "한 곡선으로 판단하지 마라".
아래 셀로 본문과 1:1 대조한다.

In [5]:
res0 = results[0]
print(f"first20 mean = {np.mean(res0['rets'][:20]):.2f}   (본문: 21.15)")
print(f"last20 mean  = {np.mean(res0['rets'][-20:]):.2f}   (본문: 103.85)")
print(f"best         = {res0['best']:.0f}   (본문: 500)")
for ep in (0, 50, 100, 150, 200, 250):
    print(f"  episode {ep}: {res0['rets'][ep]:.0f}")


first20 mean = 21.15   (본문: 21.15)
last20 mean  = 103.85   (본문: 103.85)
best         = 500   (본문: 500)
  episode 0: 9
  episode 50: 11
  episode 100: 66
  episode 150: 120
  episode 200: 259
  episode 250: 500


## 4. 학습곡선 그리기 (본문 그림 (a)(b))

왼쪽: 에피소드 리턴(얇은 선=개별 에피소드, 두꺼운 선=10-에피소드 이동평균,
3시드). 오른쪽: TD 손실(로그 스케일, 200스텝 이동평균). 손실 곡선이
"자기(타겟) 목표와의 일관성"을 재는 지표이지 참값과의 거리를 재는 것이
아님을 기억하자(9.2절).

In [6]:
IMG = "/home/smhan/book-ml/kor/src/images"
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
for seed, color in ((0, "#1f77b4"), (1, "#d62728"), (2, "#2ca02c")):
    rets = results[seed]["rets"]
    sm = np.convolve(rets, np.ones(10) / 10, mode="valid")
    ax.plot(np.arange(1, 301), rets, color=color, alpha=0.15, lw=0.8)
    ax.plot(np.arange(10, 301), sm, color=color, lw=1.8, label=f"시드 {seed} (10-에피소드 이동평균)")
ax.axhline(500, color="k", ls=":", lw=1)
ax.text(299, 460, "500 (만점)", ha="right", fontsize=8, color="k")
ax.set_xlabel("에피소드")
ax.set_ylabel("에피소드 리턴 (스텝 수)")
ax.set_title("CartPole DQN 학습곡선 — 시드 3개")
ax.legend(fontsize=8, loc="lower right")
ax.set_xlim(0, 301); ax.set_ylim(0, 520)

ax = axes[1]
losses = res0["losses"]
w = 200
smloss = np.convolve(losses, np.ones(w) / w, mode="valid")
ax.plot(np.arange(len(losses)), losses, color="#7f7f7f", alpha=0.15, lw=0.6)
ax.plot(np.arange(w, len(losses) + 1), smloss, color="#8c564b", lw=1.8,
        label="TD 손실 (200-스텝 이동평균, 시드 0)")
ax.set_yscale("log")
ax.set_xlabel("학습 스텝 (버퍼 채움 이후)")
ax.set_ylabel("TD 손실 (log)")
ax.set_title("DQN 손실 곡선")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig(IMG + "/ch09_3_dqn_cartpole_curve.svg", bbox_inches="tight")
plt.show()
print("saved:", IMG + "/ch09_3_dqn_cartpole_curve.svg")


saved: /home/smhan/book-ml/kor/src/images/ch09_3_dqn_cartpole_curve.svg


## 5. 학습이 "진짜"였는지 확인: 탐색 없이 그리디 플레이

학습된 네트워크(`시드 0`)로 `ε=0`(탐색 OFF, argmax만)으로 10에피소드를
돌린다. "학습곡선이 올랐다"는 것(탐색이 섞인 훈련 리턴)과 달리,
**그리디 평가 리턴**이 높게 나와야 "에이전트가 실제로 막대를 세우는
방법을 익혔다"는 증거가 된다. 본문 그림 (a)의 이동평균이 수렴하는
높이와 같은 스케일(수백)이어야 한다.

In [7]:
def evaluate(Q_net, seed, n_eps=10):
    env = gym.make("CartPole-v1")
    rets = []
    for i in range(n_eps):
        obs, _ = env.reset(seed=seed + 7777 * i)
        done, ret = False, 0
        while not done:
            with torch.no_grad():
                a = int(Q_net(torch.tensor(obs, dtype=torch.float32)).argmax())
            obs, r, term, trunc, _ = env.step(a)
            done = term or trunc
            ret += r
        rets.append(ret)
    env.close()
    return rets

eval_rets = evaluate(results[0]["Q_net"], seed=12345, n_eps=10)
print("그리디(ε=0) 10에피소드:", [int(r) for r in eval_rets])
print(f"평균 = {np.mean(eval_rets):.0f}, 최고 = {np.max(eval_rets):.0f} (만점 500)")


그리디(ε=0) 10에피소드: [106, 109, 106, 102, 102, 104, 107, 101, 103, 100]
평균 = 104, 최고 = 109 (만점 500)


## 6. 정리

- **경험 재현**은 "데이터" 자리에 지도학습의 전제(독립적 샘플)를 인위적으로
  되살린다 — FIFO 버퍼 + 무작위 샘플링의 두 줄짜리 장치.
- **타겟 네트워크**는 "정답" 자리에(목표값을 `C`스텝 동결) 되살린다.
- 둘이 함께야 CartPole 학습곡선이 (시드마다 높이는 달라도) **일관되게
  오른다** — 하지만 들쭉날쭉함(진동)은 남으므로, **여러 시드의 평균
  곡선**과 **그리디 평가 리턴**으로 판단한다.
- 다음 장(Chapter 10~11)에서는 Q값을 거치지 않고 **정책을 직접** 학습하는
  actor-critic 계열로 넘어간다.